# 📊 Análisis Exploratorio de Datos (EDA) - Heart Disease Prediction

**Autor:** Miembro P del Equipo

**Objetivo:** Realizar un análisis exploratorio exhaustivo del dataset de enfermedades cardíacas para responder preguntas de investigación clave y preparar los datos para el modelado predictivo.

---

## 📋 Índice

1. **Importación de Librerías y Carga de Datos**
2. **Exploración General del Dataset**
3. **Análisis de Variables Individuales**
4. **Análisis de Relaciones entre Variables**
5. **Detección de Anomalías**
6. **Conclusiones y Recomendaciones**

---
## 1️⃣ Importación de Librerías y Carga de Datos

In [ ]:
# Librerías básicas
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

# Análisis estadístico
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Configuración de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print('✅ Librerías importadas correctamente')

In [ ]:
# Cargar el dataset
df = pd.read_csv('../data/heart.csv')

print(f"📊 Dataset cargado exitosamente")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"\nPrimeras filas:")
df.head(10)

### 📖 Descripción de Variables

| Variable | Descripción | Tipo |
|----------|-------------|------|
| **Age** | Edad del paciente | Numérica |
| **Sex** | Sexo (M: Masculino, F: Femenino) | Categórica |
| **ChestPainType** | Tipo de dolor torácico (ATA, NAP, ASY, TA) | Categórica |
| **RestingBP** | Presión arterial en reposo (mm Hg) | Numérica |
| **Cholesterol** | Colesterol sérico (mg/dl) | Numérica |
| **FastingBS** | Azúcar en sangre en ayunas > 120 mg/dl (1: Sí, 0: No) | Binaria |
| **RestingECG** | Resultados electrocardiográficos en reposo | Categórica |
| **MaxHR** | Frecuencia cardíaca máxima alcanzada | Numérica |
| **ExerciseAngina** | Angina inducida por ejercicio (Y: Sí, N: No) | Binaria |
| **Oldpeak** | Depresión del ST | Numérica |
| **ST_Slope** | Pendiente del segmento ST | Categórica |
| **HeartDisease** | Variable objetivo (1: Enfermedad, 0: Sin enfermedad) | Binaria |

---
## 2️⃣ Exploración General del Dataset

### 🔍 Pregunta 1: ¿Cuántos pacientes hay con y sin enfermedad cardíaca? (Balance de clases)

In [ ]:
# Información general del dataset
print("="*80)
print("INFORMACIÓN GENERAL DEL DATASET")
print("="*80)
df.info()

In [ ]:
# Balance de clases
print("\n" + "="*80)
print("BALANCE DE CLASES - HeartDisease")
print("="*80)

# Conteo absoluto y porcentual
class_counts = df['HeartDisease'].value_counts()
class_pct = df['HeartDisease'].value_counts(normalize=True) * 100

balance_df = pd.DataFrame({
    'Cantidad': class_counts,
    'Porcentaje (%)': class_pct
})
balance_df.index = ['Sin Enfermedad (0)', 'Con Enfermedad (1)']

print(balance_df)
print(f"\n📊 Ratio de balance: {class_counts[1]/class_counts[0]:.2f}")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(balance_df.index, balance_df['Cantidad'], color=colors, alpha=0.8, edgecolor='black')
axes[0].set_title('Distribución de Clases - Enfermedad Cardíaca', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Número de Pacientes', fontsize=12)
axes[0].grid(axis='y', alpha=0.3)
for i, (idx, val) in enumerate(balance_df['Cantidad'].items()):
    axes[0].text(i, val + 10, str(val), ha='center', fontsize=11, fontweight='bold')

# Gráfico de pastel
axes[1].pie(balance_df['Cantidad'], labels=balance_df.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.05, 0.05))
axes[1].set_title('Proporción de Pacientes', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Interpretación
if abs(class_pct[0] - class_pct[1]) < 10:
    print("\n✅ El dataset está BALANCEADO (diferencia < 10%)")
elif abs(class_pct[0] - class_pct[1]) < 30:
    print("\n⚠️ El dataset está LIGERAMENTE DESBALANCEADO")
else:
    print("\n❌ El dataset está DESBALANCEADO - Se requieren técnicas de balanceo")

### 🔍 Pregunta 2: ¿Cuáles son las características demográficas de la población? (edad, sexo)

In [ ]:
print("="*80)
print("CARACTERÍSTICAS DEMOGRÁFICAS")
print("="*80)

# Estadísticas de edad
print("\n📊 EDAD:")
print(df['Age'].describe())
print(f"\n  • Edad mínima: {df['Age'].min()} años")
print(f"  • Edad máxima: {df['Age'].max()} años")
print(f"  • Edad promedio: {df['Age'].mean():.1f} años")
print(f"  • Edad mediana: {df['Age'].median():.1f} años")

# Distribución por sexo
print("\n👥 SEXO:")
sex_counts = df['Sex'].value_counts()
sex_pct = df['Sex'].value_counts(normalize=True) * 100
for sex, count in sex_counts.items():
    pct = sex_pct[sex]
    label = 'Masculino' if sex == 'M' else 'Femenino'
    print(f"  • {label}: {count} ({pct:.1f}%)")

# Visualización
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Distribución de edad
axes[0, 0].hist(df['Age'], bins=30, color='#3498db', alpha=0.7, edgecolor='black')
axes[0, 0].axvline(df['Age'].mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {df["Age"].mean():.1f}')
axes[0, 0].axvline(df['Age'].median(), color='green', linestyle='--', linewidth=2, label=f'Mediana: {df["Age"].median():.1f}')
axes[0, 0].set_title('Distribución de Edad', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Edad (años)')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Boxplot de edad
axes[0, 1].boxplot(df['Age'], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#3498db', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2))
axes[0, 1].set_title('Boxplot de Edad', fontsize=13, fontweight='bold')
axes[0, 1].set_ylabel('Edad (años)')
axes[0, 1].grid(alpha=0.3)

# 3. Distribución por sexo
sex_colors = ['#e91e63', '#2196f3']
sex_labels = ['Femenino', 'Masculino']
sex_data = [sex_counts.get('F', 0), sex_counts.get('M', 0)]
axes[1, 0].bar(sex_labels, sex_data, color=sex_colors, alpha=0.8, edgecolor='black')
axes[1, 0].set_title('Distribución por Sexo', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Número de Pacientes')
axes[1, 0].grid(axis='y', alpha=0.3)
for i, val in enumerate(sex_data):
    axes[1, 0].text(i, val + 10, str(val), ha='center', fontsize=11, fontweight='bold')

# 4. Edad por sexo
df.boxplot(column='Age', by='Sex', ax=axes[1, 1], patch_artist=True)
axes[1, 1].set_title('Distribución de Edad por Sexo', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Sexo')
axes[1, 1].set_ylabel('Edad (años)')
plt.suptitle('')  # Remover título automático

plt.tight_layout()
plt.show()

### 🔍 Pregunta 3: ¿Existen datos faltantes o inconsistencias que requieran tratamiento?

In [ ]:
print("="*80)
print("ANÁLISIS DE DATOS FALTANTES")
print("="*80)

# Valores nulos
missing_data = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Valores Faltantes': missing_data,
    'Porcentaje (%)': missing_pct
})
missing_df = missing_df[missing_df['Valores Faltantes'] > 0].sort_values('Valores Faltantes', ascending=False)

if len(missing_df) > 0:
    print("\n⚠️ Se encontraron datos faltantes:")
    print(missing_df)
else:
    print("\n✅ No hay datos faltantes en el dataset")

# Visualización de datos faltantes
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Matriz de valores faltantes
msno.matrix(df, ax=axes[0], fontsize=10, sparkline=False)
axes[0].set_title('Matriz de Valores Faltantes', fontsize=13, fontweight='bold')

# Heatmap de completitud
msno.heatmap(df, ax=axes[1], fontsize=10)
axes[1].set_title('Correlación de Completitud', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Análisis de valores anómalos en variables numéricas
print("\n" + "="*80)
print("ANÁLISIS DE INCONSISTENCIAS")
print("="*80)

# Verificar valores de 0 en Colesterol (posible inconsistencia)
zero_cholesterol = (df['Cholesterol'] == 0).sum()
print(f"\n⚠️ Valores de Colesterol = 0: {zero_cholesterol} ({zero_cholesterol/len(df)*100:.2f}%)")
if zero_cholesterol > 0:
    print("   → Esto puede indicar datos faltantes codificados como 0")

# Verificar valores de 0 en RestingBP
zero_bp = (df['RestingBP'] == 0).sum()
print(f"\n⚠️ Valores de RestingBP = 0: {zero_bp} ({zero_bp/len(df)*100:.2f}%)")
if zero_bp > 0:
    print("   → Esto puede indicar datos faltantes codificados como 0")

# Verificar duplicados
duplicates = df.duplicated().sum()
print(f"\n🔍 Filas duplicadas: {duplicates}")
if duplicates > 0:
    print("   → Se recomienda revisar y eliminar duplicados")

---
## 3️⃣ Análisis de Variables Individuales

### 🔍 Pregunta 4: ¿Cómo se distribuye la edad en pacientes con y sin enfermedad?

In [ ]:
# Estadísticas de edad por grupo
print("="*80)
print("DISTRIBUCIÓN DE EDAD POR DIAGNÓSTICO")
print("="*80)

age_by_disease = df.groupby('HeartDisease')['Age'].describe()
age_by_disease.index = ['Sin Enfermedad', 'Con Enfermedad']
print("\n", age_by_disease)

# Test estadístico
age_healthy = df[df['HeartDisease'] == 0]['Age']
age_sick = df[df['HeartDisease'] == 1]['Age']
stat, p_value = mannwhitneyu(age_healthy, age_sick)
print(f"\n📊 Test de Mann-Whitney U:")
print(f"   • Estadístico: {stat:.2f}")
print(f"   • p-valor: {p_value:.4f}")
if p_value < 0.05:
    print("   ✅ HAY diferencia significativa en edad entre grupos (p < 0.05)")
else:
    print("   ❌ NO hay diferencia significativa en edad entre grupos (p >= 0.05)")

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogramas superpuestos
axes[0].hist(age_healthy, bins=20, alpha=0.6, label='Sin Enfermedad', color='#2ecc71', edgecolor='black')
axes[0].hist(age_sick, bins=20, alpha=0.6, label='Con Enfermedad', color='#e74c3c', edgecolor='black')
axes[0].set_title('Distribución de Edad por Diagnóstico', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Edad (años)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Boxplot comparativo
df.boxplot(column='Age', by='HeartDisease', ax=axes[1],
           patch_artist=True, positions=[0, 1])
axes[1].set_title('Boxplot de Edad por Diagnóstico', fontsize=13, fontweight='bold')
axes[1].set_xlabel('HeartDisease (0: No, 1: Sí)')
axes[1].set_ylabel('Edad (años)')
plt.suptitle('')

# Violinplot
sns.violinplot(data=df, x='HeartDisease', y='Age', ax=axes[2], palette=['#2ecc71', '#e74c3c'])
axes[2].set_title('Violinplot de Edad por Diagnóstico', fontsize=13, fontweight='bold')
axes[2].set_xlabel('HeartDisease (0: No, 1: Sí)')
axes[2].set_ylabel('Edad (años)')

plt.tight_layout()
plt.show()

### 🔍 Pregunta 5: ¿Qué tipo de dolor torácico es más común en pacientes enfermos?

In [ ]:
print("="*80)
print("ANÁLISIS DE TIPO DE DOLOR TORÁCICO")
print("="*80)

# Tabla cruzada
chest_pain_cross = pd.crosstab(df['ChestPainType'], df['HeartDisease'], 
                                normalize='columns') * 100
chest_pain_cross.columns = ['Sin Enfermedad (%)', 'Con Enfermedad (%)']
print("\n📊 Distribución de Tipo de Dolor por Diagnóstico:")
print(chest_pain_cross.round(2))

# Test Chi-cuadrado
contingency_table = pd.crosstab(df['ChestPainType'], df['HeartDisease'])
chi2, p_val, dof, expected = chi2_contingency(contingency_table)
print(f"\n📊 Test Chi-cuadrado:")
print(f"   • Chi2: {chi2:.2f}")
print(f"   • p-valor: {p_val:.6f}")
if p_val < 0.05:
    print("   ✅ HAY asociación significativa entre tipo de dolor y enfermedad (p < 0.05)")
else:
    print("   ❌ NO hay asociación significativa (p >= 0.05)")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de barras agrupadas
chest_pain_counts = pd.crosstab(df['ChestPainType'], df['HeartDisease'])
chest_pain_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'], 
                       alpha=0.8, edgecolor='black')
axes[0].set_title('Tipo de Dolor Torácico por Diagnóstico', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Tipo de Dolor Torácico')
axes[0].set_ylabel('Número de Pacientes')
axes[0].legend(['Sin Enfermedad', 'Con Enfermedad'])
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Heatmap de proporciones
sns.heatmap(chest_pain_cross.T, annot=True, fmt='.1f', cmap='RdYlGn_r', 
            ax=axes[1], cbar_kws={'label': 'Porcentaje (%)'})
axes[1].set_title('Proporción de Tipo de Dolor por Diagnóstico', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Diagnóstico')
axes[1].set_xlabel('Tipo de Dolor Torácico')

plt.tight_layout()
plt.show()

# Tipos de dolor torácico explicados
print("\n📖 Tipos de Dolor Torácico:")
print("   • ATA: Angina Típica")
print("   • NAP: Dolor No Anginoso")
print("   • ASY: Asintomático")
print("   • TA: Angina Atípica")

### 🔍 Pregunta 6: ¿Existe diferencia en los niveles de colesterol entre ambos grupos?

In [ ]:
print("="*80)
print("ANÁLISIS DE COLESTEROL")
print("="*80)

# Filtrar colesterol > 0 (eliminar posibles valores faltantes codificados como 0)
df_cholesterol = df[df['Cholesterol'] > 0]

print(f"\n📊 Pacientes con colesterol válido (>0): {len(df_cholesterol)} de {len(df)}")
print(f"   Excluidos: {len(df) - len(df_cholesterol)} pacientes con colesterol = 0\n")

# Estadísticas por grupo
chol_by_disease = df_cholesterol.groupby('HeartDisease')['Cholesterol'].describe()
chol_by_disease.index = ['Sin Enfermedad', 'Con Enfermedad']
print(chol_by_disease)

# Test estadístico
chol_healthy = df_cholesterol[df_cholesterol['HeartDisease'] == 0]['Cholesterol']
chol_sick = df_cholesterol[df_cholesterol['HeartDisease'] == 1]['Cholesterol']
stat, p_value = mannwhitneyu(chol_healthy, chol_sick)
print(f"\n📊 Test de Mann-Whitney U:")
print(f"   • p-valor: {p_value:.4f}")
if p_value < 0.05:
    print("   ✅ HAY diferencia significativa en colesterol entre grupos")
else:
    print("   ❌ NO hay diferencia significativa en colesterol entre grupos")

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogramas
axes[0].hist(chol_healthy, bins=30, alpha=0.6, label='Sin Enfermedad', color='#2ecc71', edgecolor='black')
axes[0].hist(chol_sick, bins=30, alpha=0.6, label='Con Enfermedad', color='#e74c3c', edgecolor='black')
axes[0].axvline(chol_healthy.mean(), color='#2ecc71', linestyle='--', linewidth=2)
axes[0].axvline(chol_sick.mean(), color='#e74c3c', linestyle='--', linewidth=2)
axes[0].set_title('Distribución de Colesterol por Diagnóstico', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Colesterol (mg/dl)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Boxplot
df_cholesterol.boxplot(column='Cholesterol', by='HeartDisease', ax=axes[1],
                       patch_artist=True)
axes[1].set_title('Boxplot de Colesterol por Diagnóstico', fontsize=13, fontweight='bold')
axes[1].set_xlabel('HeartDisease (0: No, 1: Sí)')
axes[1].set_ylabel('Colesterol (mg/dl)')
plt.suptitle('')

# Violinplot
sns.violinplot(data=df_cholesterol, x='HeartDisease', y='Cholesterol', 
               ax=axes[2], palette=['#2ecc71', '#e74c3c'])
axes[2].set_title('Violinplot de Colesterol por Diagnóstico', fontsize=13, fontweight='bold')
axes[2].set_xlabel('HeartDisease (0: No, 1: Sí)')
axes[2].set_ylabel('Colesterol (mg/dl)')

plt.tight_layout()
plt.show()

### 🔍 Pregunta 7: ¿La presión arterial en reposo es un factor diferenciador?

In [ ]:
print("="*80)
print("ANÁLISIS DE PRESIÓN ARTERIAL EN REPOSO")
print("="*80)

# Filtrar valores válidos
df_bp = df[df['RestingBP'] > 0]

# Estadísticas
bp_by_disease = df_bp.groupby('HeartDisease')['RestingBP'].describe()
bp_by_disease.index = ['Sin Enfermedad', 'Con Enfermedad']
print("\n", bp_by_disease)

# Test estadístico
bp_healthy = df_bp[df_bp['HeartDisease'] == 0]['RestingBP']
bp_sick = df_bp[df_bp['HeartDisease'] == 1]['RestingBP']
stat, p_value = mannwhitneyu(bp_healthy, bp_sick)
print(f"\n📊 Test de Mann-Whitney U:")
print(f"   • p-valor: {p_value:.4f}")
if p_value < 0.05:
    print("   ✅ HAY diferencia significativa en presión arterial")
else:
    print("   ❌ NO hay diferencia significativa en presión arterial")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
sns.boxplot(data=df_bp, x='HeartDisease', y='RestingBP', ax=axes[0],
            palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Presión Arterial en Reposo por Diagnóstico', fontsize=13, fontweight='bold')
axes[0].set_xlabel('HeartDisease (0: No, 1: Sí)')
axes[0].set_ylabel('RestingBP (mm Hg)')

# Violinplot
sns.violinplot(data=df_bp, x='HeartDisease', y='RestingBP', ax=axes[1],
               palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Distribución de Presión Arterial', fontsize=13, fontweight='bold')
axes[1].set_xlabel('HeartDisease (0: No, 1: Sí)')
axes[1].set_ylabel('RestingBP (mm Hg)')

plt.tight_layout()
plt.show()

### 🔍 Pregunta 8: ¿Cómo afecta el azúcar en sangre a la predicción?

In [ ]:
print("="*80)
print("ANÁLISIS DE AZÚCAR EN SANGRE EN AYUNAS")
print("="*80)

# Tabla cruzada
fasting_bs_cross = pd.crosstab(df['FastingBS'], df['HeartDisease'], margins=True)
print("\n📊 Tabla de Contingencia - FastingBS vs HeartDisease:")
print(fasting_bs_cross)

# Porcentajes
fasting_bs_pct = pd.crosstab(df['FastingBS'], df['HeartDisease'], normalize='columns') * 100
fasting_bs_pct.columns = ['Sin Enfermedad (%)', 'Con Enfermedad (%)']
fasting_bs_pct.index = ['FastingBS ≤ 120 mg/dl', 'FastingBS > 120 mg/dl']
print("\n", fasting_bs_pct.round(2))

# Test Chi-cuadrado
chi2, p_val, dof, expected = chi2_contingency(pd.crosstab(df['FastingBS'], df['HeartDisease']))
print(f"\n📊 Test Chi-cuadrado:")
print(f"   • p-valor: {p_val:.4f}")
if p_val < 0.05:
    print("   ✅ HAY asociación significativa entre azúcar en sangre y enfermedad")
else:
    print("   ❌ NO hay asociación significativa")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras agrupadas
fasting_bs_counts = pd.crosstab(df['FastingBS'], df['HeartDisease'])
fasting_bs_counts.index = ['≤ 120 mg/dl', '> 120 mg/dl']
fasting_bs_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'],
                       alpha=0.8, edgecolor='black')
axes[0].set_title('Distribución de Azúcar en Sangre por Diagnóstico', fontsize=13, fontweight='bold')
axes[0].set_xlabel('FastingBS')
axes[0].set_ylabel('Número de Pacientes')
axes[0].legend(['Sin Enfermedad', 'Con Enfermedad'])
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Proporción normalizada
fasting_bs_norm = pd.crosstab(df['FastingBS'], df['HeartDisease'], normalize='index') * 100
fasting_bs_norm.index = ['≤ 120 mg/dl', '> 120 mg/dl']
fasting_bs_norm.plot(kind='bar', stacked=True, ax=axes[1], 
                     color=['#2ecc71', '#e74c3c'], alpha=0.8, edgecolor='black')
axes[1].set_title('Proporción de Diagnóstico por Nivel de Azúcar', fontsize=13, fontweight='bold')
axes[1].set_xlabel('FastingBS')
axes[1].set_ylabel('Porcentaje (%)')
axes[1].legend(['Sin Enfermedad', 'Con Enfermedad'])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

### 🔍 Pregunta 9: ¿Hay diferencias en la frecuencia cardíaca máxima alcanzada?

In [ ]:
print("="*80)
print("ANÁLISIS DE FRECUENCIA CARDÍACA MÁXIMA")
print("="*80)

# Estadísticas
maxhr_by_disease = df.groupby('HeartDisease')['MaxHR'].describe()
maxhr_by_disease.index = ['Sin Enfermedad', 'Con Enfermedad']
print("\n", maxhr_by_disease)

# Test estadístico
maxhr_healthy = df[df['HeartDisease'] == 0]['MaxHR']
maxhr_sick = df[df['HeartDisease'] == 1]['MaxHR']
stat, p_value = mannwhitneyu(maxhr_healthy, maxhr_sick)
print(f"\n📊 Test de Mann-Whitney U:")
print(f"   • p-valor: {p_value:.6f}")
if p_value < 0.05:
    print("   ✅ HAY diferencia significativa en MaxHR entre grupos")
    print(f"   → Media Sin Enfermedad: {maxhr_healthy.mean():.1f} bpm")
    print(f"   → Media Con Enfermedad: {maxhr_sick.mean():.1f} bpm")
else:
    print("   ❌ NO hay diferencia significativa")

# Visualización
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Histogramas
axes[0, 0].hist(maxhr_healthy, bins=25, alpha=0.6, label='Sin Enfermedad', color='#2ecc71', edgecolor='black')
axes[0, 0].hist(maxhr_sick, bins=25, alpha=0.6, label='Con Enfermedad', color='#e74c3c', edgecolor='black')
axes[0, 0].set_title('Distribución de Frecuencia Cardíaca Máxima', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('MaxHR (bpm)')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Boxplot
sns.boxplot(data=df, x='HeartDisease', y='MaxHR', ax=axes[0, 1],
            palette=['#2ecc71', '#e74c3c'])
axes[0, 1].set_title('Boxplot de MaxHR por Diagnóstico', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('HeartDisease (0: No, 1: Sí)')
axes[0, 1].set_ylabel('MaxHR (bpm)')

# Violinplot
sns.violinplot(data=df, x='HeartDisease', y='MaxHR', ax=axes[1, 0],
               palette=['#2ecc71', '#e74c3c'])
axes[1, 0].set_title('Violinplot de MaxHR', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('HeartDisease (0: No, 1: Sí)')
axes[1, 0].set_ylabel('MaxHR (bpm)')

# KDE
maxhr_healthy.plot(kind='kde', ax=axes[1, 1], label='Sin Enfermedad', color='#2ecc71', linewidth=2)
maxhr_sick.plot(kind='kde', ax=axes[1, 1], label='Con Enfermedad', color='#e74c3c', linewidth=2)
axes[1, 1].set_title('Densidad de Frecuencia Cardíaca Máxima', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('MaxHR (bpm)')
axes[1, 1].set_ylabel('Densidad')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4️⃣ Análisis de Relaciones entre Variables

### 🔍 Pregunta 10: ¿Qué variables tienen mayor correlación con la enfermedad cardíaca?

In [ ]:
print("="*80)
print("MATRIZ DE CORRELACIÓN")
print("="*80)

# Crear copia con variables codificadas
df_encoded = df.copy()

# Codificar variables categóricas
df_encoded['Sex'] = df_encoded['Sex'].map({'M': 1, 'F': 0})
df_encoded['ExerciseAngina'] = df_encoded['ExerciseAngina'].map({'Y': 1, 'N': 0})
df_encoded = pd.get_dummies(df_encoded, columns=['ChestPainType', 'RestingECG', 'ST_Slope'], drop_first=True)

# Matriz de correlación
correlation_matrix = df_encoded.corr()

# Correlaciones con HeartDisease
heart_disease_corr = correlation_matrix['HeartDisease'].sort_values(ascending=False)
print("\n📊 Correlaciones con HeartDisease (ordenadas):")
print(heart_disease_corr)

# Top 10 variables más correlacionadas
top_corr = heart_disease_corr.drop('HeartDisease').abs().sort_values(ascending=False).head(10)
print("\n🔝 Top 10 variables con mayor correlación (valor absoluto):")
for var, corr in top_corr.items():
    print(f"   • {var}: {heart_disease_corr[var]:.3f}")

In [ ]:
# Visualización de matriz de correlación
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Heatmap completo
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0,
            ax=axes[0], cbar_kws={'label': 'Correlación'})
axes[0].set_title('Matriz de Correlación Completa', fontsize=14, fontweight='bold')

# Heatmap de correlaciones con HeartDisease
heart_corr_df = heart_disease_corr.drop('HeartDisease').to_frame()
sns.heatmap(heart_corr_df, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            ax=axes[1], cbar_kws={'label': 'Correlación'}, vmin=-1, vmax=1)
axes[1].set_title('Correlaciones con HeartDisease', fontsize=14, fontweight='bold')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Gráfico de barras de top correlaciones
fig, ax = plt.subplots(figsize=(12, 8))

top_15_corr = heart_disease_corr.drop('HeartDisease').abs().sort_values(ascending=True).tail(15)
colors_bar = ['#e74c3c' if heart_disease_corr[var] > 0 else '#3498db' for var in top_15_corr.index]

top_15_values = [heart_disease_corr[var] for var in top_15_corr.index]
ax.barh(range(len(top_15_corr)), top_15_values, color=colors_bar, alpha=0.8, edgecolor='black')
ax.set_yticks(range(len(top_15_corr)))
ax.set_yticklabels(top_15_corr.index)
ax.set_xlabel('Correlación con HeartDisease', fontsize=12)
ax.set_title('Top 15 Variables con Mayor Correlación', fontsize=14, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)
ax.grid(axis='x', alpha=0.3)

# Añadir valores en las barras
for i, val in enumerate(top_15_values):
    ax.text(val, i, f' {val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### 🔍 Pregunta 11: ¿Existen patrones diferentes entre hombres y mujeres?

In [ ]:
print("="*80)
print("ANÁLISIS POR SEXO")
print("="*80)

# Tasa de enfermedad por sexo
sex_disease = pd.crosstab(df['Sex'], df['HeartDisease'], normalize='index') * 100
sex_disease.columns = ['Sin Enfermedad (%)', 'Con Enfermedad (%)']
sex_disease.index = ['Femenino', 'Masculino']
print("\n📊 Tasa de Enfermedad por Sexo:")
print(sex_disease.round(2))

# Estadísticas clínicas por sexo y enfermedad
print("\n📊 Edad Promedio por Sexo y Diagnóstico:")
age_sex_disease = df.groupby(['Sex', 'HeartDisease'])['Age'].mean().unstack()
age_sex_disease.columns = ['Sin Enfermedad', 'Con Enfermedad']
age_sex_disease.index = ['Femenino', 'Masculino']
print(age_sex_disease.round(1))

print("\n📊 Colesterol Promedio por Sexo y Diagnóstico:")
df_chol_valid = df[df['Cholesterol'] > 0]
chol_sex_disease = df_chol_valid.groupby(['Sex', 'HeartDisease'])['Cholesterol'].mean().unstack()
chol_sex_disease.columns = ['Sin Enfermedad', 'Con Enfermedad']
chol_sex_disease.index = ['Femenino', 'Masculino']
print(chol_sex_disease.round(1))

In [ ]:
# Visualización completa por sexo
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Tasa de enfermedad por sexo
sex_disease_counts = pd.crosstab(df['Sex'], df['HeartDisease'])
sex_disease_counts.index = ['Femenino', 'Masculino']
sex_disease_counts.plot(kind='bar', ax=axes[0, 0], color=['#2ecc71', '#e74c3c'],
                        alpha=0.8, edgecolor='black')
axes[0, 0].set_title('Distribución de Enfermedad por Sexo', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Sexo')
axes[0, 0].set_ylabel('Número de Pacientes')
axes[0, 0].legend(['Sin Enfermedad', 'Con Enfermedad'])
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=45)
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Edad por sexo
df.boxplot(column='Age', by='Sex', ax=axes[0, 1], patch_artist=True)
axes[0, 1].set_title('Edad por Sexo', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Sexo')
axes[0, 1].set_ylabel('Edad (años)')
plt.suptitle('')

# 3. MaxHR por sexo
df.boxplot(column='MaxHR', by='Sex', ax=axes[0, 2], patch_artist=True)
axes[0, 2].set_title('Frecuencia Cardíaca Máxima por Sexo', fontsize=12, fontweight='bold')
axes[0, 2].set_xlabel('Sexo')
axes[0, 2].set_ylabel('MaxHR (bpm)')
plt.suptitle('')

# 4. Tipo de dolor por sexo
chest_sex = pd.crosstab(df['Sex'], df['ChestPainType'], normalize='index') * 100
chest_sex.index = ['Femenino', 'Masculino']
chest_sex.plot(kind='bar', ax=axes[1, 0], alpha=0.8, edgecolor='black')
axes[1, 0].set_title('Tipo de Dolor Torácico por Sexo', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Sexo')
axes[1, 0].set_ylabel('Porcentaje (%)')
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45)
axes[1, 0].legend(title='Tipo Dolor', bbox_to_anchor=(1.05, 1))

# 5. Angina por ejercicio por sexo y enfermedad
angina_sex = pd.crosstab([df['Sex'], df['HeartDisease']], df['ExerciseAngina'], normalize='index') * 100
angina_sex_y = angina_sex['Y'].unstack()
angina_sex_y.index = ['Femenino', 'Masculino']
angina_sex_y.columns = ['Sin Enfermedad', 'Con Enfermedad']
angina_sex_y.plot(kind='bar', ax=axes[1, 1], color=['#2ecc71', '#e74c3c'],
                  alpha=0.8, edgecolor='black')
axes[1, 1].set_title('% con Angina por Ejercicio', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Sexo')
axes[1, 1].set_ylabel('Porcentaje con Angina (%)')
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=45)
axes[1, 1].legend()

# 6. Distribución de edad por sexo y enfermedad
for disease in [0, 1]:
    for sex in ['M', 'F']:
        data = df[(df['HeartDisease'] == disease) & (df['Sex'] == sex)]['Age']
        label = f"{'M' if sex == 'M' else 'F'} - {'Enf.' if disease == 1 else 'Sano'}"
        color = '#e74c3c' if disease == 1 else '#2ecc71'
        linestyle = '-' if sex == 'M' else '--'
        data.plot(kind='kde', ax=axes[1, 2], label=label, color=color, 
                  linestyle=linestyle, linewidth=2, alpha=0.7)
axes[1, 2].set_title('Densidad de Edad por Sexo y Diagnóstico', fontsize=12, fontweight='bold')
axes[1, 2].set_xlabel('Edad (años)')
axes[1, 2].set_ylabel('Densidad')
axes[1, 2].legend()
axes[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 🔍 Pregunta 12: ¿La combinación de edad + colesterol alto incrementa el riesgo?

In [ ]:
print("="*80)
print("ANÁLISIS DE INTERACCIÓN: EDAD + COLESTEROL")
print("="*80)

# Crear grupos de edad y colesterol
df_valid = df[df['Cholesterol'] > 0].copy()

# Categorizar edad
df_valid['AgeGroup'] = pd.cut(df_valid['Age'], bins=[0, 40, 50, 60, 100],
                               labels=['<40', '40-50', '50-60', '>60'])

# Categorizar colesterol (según guías médicas)
df_valid['CholesterolLevel'] = pd.cut(df_valid['Cholesterol'], 
                                       bins=[0, 200, 240, 1000],
                                       labels=['Normal (<200)', 'Límite (200-240)', 'Alto (>240)'])

# Tabla de contingencia
print("\n📊 Tasa de Enfermedad por Grupo de Edad y Nivel de Colesterol:\n")
age_chol_disease = df_valid.groupby(['AgeGroup', 'CholesterolLevel'])['HeartDisease'].agg(['mean', 'count'])
age_chol_disease['mean'] = age_chol_disease['mean'] * 100
age_chol_disease.columns = ['Tasa Enfermedad (%)', 'N Pacientes']
print(age_chol_disease.round(2))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot: Edad vs Colesterol coloreado por enfermedad
scatter_healthy = df_valid[df_valid['HeartDisease'] == 0]
scatter_sick = df_valid[df_valid['HeartDisease'] == 1]

axes[0].scatter(scatter_healthy['Age'], scatter_healthy['Cholesterol'], 
                c='#2ecc71', alpha=0.5, s=50, label='Sin Enfermedad', edgecolors='black', linewidth=0.5)
axes[0].scatter(scatter_sick['Age'], scatter_sick['Cholesterol'], 
                c='#e74c3c', alpha=0.5, s=50, label='Con Enfermedad', edgecolors='black', linewidth=0.5)
axes[0].axhline(200, color='orange', linestyle='--', linewidth=1.5, label='Colesterol Normal')
axes[0].axhline(240, color='red', linestyle='--', linewidth=1.5, label='Colesterol Alto')
axes[0].set_title('Relación Edad-Colesterol por Diagnóstico', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Edad (años)')
axes[0].set_ylabel('Colesterol (mg/dl)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Heatmap de tasa de enfermedad
pivot_disease_rate = df_valid.pivot_table(values='HeartDisease', 
                                           index='AgeGroup', 
                                           columns='CholesterolLevel',
                                           aggfunc='mean') * 100
sns.heatmap(pivot_disease_rate, annot=True, fmt='.1f', cmap='YlOrRd',
            ax=axes[1], cbar_kws={'label': 'Tasa Enfermedad (%)'})
axes[1].set_title('Tasa de Enfermedad: Edad × Colesterol', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Nivel de Colesterol')
axes[1].set_ylabel('Grupo de Edad')

plt.tight_layout()
plt.show()

# Análisis de grupo de alto riesgo
high_risk = df_valid[(df_valid['Age'] > 55) & (df_valid['Cholesterol'] > 240)]
high_risk_rate = high_risk['HeartDisease'].mean() * 100
print(f"\n⚠️ GRUPO DE ALTO RIESGO (Edad >55 + Colesterol >240):")
print(f"   • N pacientes: {len(high_risk)}")
print(f"   • Tasa de enfermedad: {high_risk_rate:.1f}%")

### 🔍 Pregunta 13: ¿Los pacientes con diabetes tienen diferentes factores de riesgo?

In [ ]:
print("="*80)
print("ANÁLISIS DE PACIENTES CON DIABETES (FastingBS > 120)")
print("="*80)

# Tasa de enfermedad en diabéticos vs no diabéticos
diabetes_disease = df.groupby('FastingBS')['HeartDisease'].agg(['mean', 'count'])
diabetes_disease['mean'] = diabetes_disease['mean'] * 100
diabetes_disease.columns = ['Tasa Enfermedad (%)', 'N Pacientes']
diabetes_disease.index = ['No Diabético', 'Diabético']
print("\n", diabetes_disease.round(2))

# Comparación de variables clínicas
print("\n📊 Comparación de Variables Clínicas:\n")
clinical_vars = ['Age', 'Cholesterol', 'RestingBP', 'MaxHR', 'Oldpeak']
diabetes_comparison = df.groupby('FastingBS')[clinical_vars].mean()
diabetes_comparison.index = ['No Diabético', 'Diabético']
print(diabetes_comparison.round(2))

# Visualización
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Tasa de enfermedad
diabetes_disease['Tasa Enfermedad (%)'].plot(kind='bar', ax=axes[0, 0],
                                               color=['#3498db', '#e67e22'],
                                               alpha=0.8, edgecolor='black')
axes[0, 0].set_title('Tasa de Enfermedad Cardíaca', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Tasa (%)')
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=45)
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Edad
df.boxplot(column='Age', by='FastingBS', ax=axes[0, 1], patch_artist=True)
axes[0, 1].set_title('Edad: Diabético vs No Diabético', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('FastingBS (0: No, 1: Sí)')
axes[0, 1].set_ylabel('Edad (años)')
plt.suptitle('')

# 3. Colesterol
df[df['Cholesterol'] > 0].boxplot(column='Cholesterol', by='FastingBS', 
                                   ax=axes[0, 2], patch_artist=True)
axes[0, 2].set_title('Colesterol: Diabético vs No Diabético', fontsize=12, fontweight='bold')
axes[0, 2].set_xlabel('FastingBS (0: No, 1: Sí)')
axes[0, 2].set_ylabel('Colesterol (mg/dl)')
plt.suptitle('')

# 4. MaxHR
df.boxplot(column='MaxHR', by='FastingBS', ax=axes[1, 0], patch_artist=True)
axes[1, 0].set_title('MaxHR: Diabético vs No Diabético', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('FastingBS (0: No, 1: Sí)')
axes[1, 0].set_ylabel('MaxHR (bpm)')
plt.suptitle('')

# 5. Oldpeak
df.boxplot(column='Oldpeak', by='FastingBS', ax=axes[1, 1], patch_artist=True)
axes[1, 1].set_title('Oldpeak: Diabético vs No Diabético', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('FastingBS (0: No, 1: Sí)')
axes[1, 1].set_ylabel('Oldpeak')
plt.suptitle('')

# 6. Distribución de enfermedad en diabéticos por edad
diabetes_patients = df[df['FastingBS'] == 1]
if len(diabetes_patients) > 0:
    diabetes_patients.boxplot(column='Age', by='HeartDisease', ax=axes[1, 2],
                              patch_artist=True)
    axes[1, 2].set_title('Edad en Diabéticos por Diagnóstico', fontsize=12, fontweight='bold')
    axes[1, 2].set_xlabel('HeartDisease (0: No, 1: Sí)')
    axes[1, 2].set_ylabel('Edad (años)')
    plt.suptitle('')

plt.tight_layout()
plt.show()

### 🔍 Pregunta 14: ¿Hay interacciones significativas entre las variables?

In [ ]:
# Análisis de pairplot para variables numéricas clave
print("="*80)
print("ANÁLISIS DE INTERACCIONES ENTRE VARIABLES")
print("="*80)

# Seleccionar variables numéricas clave
key_vars = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak', 'HeartDisease']
df_pairplot = df[df['Cholesterol'] > 0][key_vars].copy()

print("\n📊 Generando Pairplot de variables clave...")
print("   (Esto puede tomar unos momentos)\n")

# Pairplot
pairplot = sns.pairplot(df_pairplot, hue='HeartDisease', 
                        palette=['#2ecc71', '#e74c3c'],
                        diag_kind='kde', plot_kws={'alpha': 0.6, 's': 30, 'edgecolor': 'k'},
                        height=2.5)
pairplot.fig.suptitle('Pairplot de Variables Numéricas Clave', 
                      fontsize=16, fontweight='bold', y=1.01)
plt.show()

print("✅ Pairplot generado")

In [ ]:
# Análisis de variables categóricas combinadas
print("\n📊 Análisis de Variables Categóricas Combinadas:\n")

# ExerciseAngina + ChestPainType
angina_pain = pd.crosstab([df['ExerciseAngina'], df['ChestPainType']], 
                          df['HeartDisease'], normalize='index') * 100
print("Tasa de Enfermedad por Angina de Ejercicio + Tipo de Dolor:")
print(angina_pain.round(2).head(10))

# Visualización de interacciones categóricas
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Angina + Tipo de dolor
interaction_data = df.groupby(['ExerciseAngina', 'ChestPainType'])['HeartDisease'].mean() * 100
interaction_data = interaction_data.unstack()
interaction_data.index = ['Sin Angina', 'Con Angina']
interaction_data.plot(kind='bar', ax=axes[0], alpha=0.8, edgecolor='black')
axes[0].set_title('Tasa de Enfermedad: Angina × Tipo de Dolor', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Tasa de Enfermedad (%)')
axes[0].set_xlabel('Angina por Ejercicio')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)
axes[0].legend(title='Tipo Dolor', bbox_to_anchor=(1.05, 1))
axes[0].grid(axis='y', alpha=0.3)

# 2. ST_Slope + ExerciseAngina
st_angina = df.groupby(['ST_Slope', 'ExerciseAngina'])['HeartDisease'].mean() * 100
st_angina = st_angina.unstack()
st_angina.columns = ['Sin Angina', 'Con Angina']
st_angina.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'],
               alpha=0.8, edgecolor='black')
axes[1].set_title('Tasa de Enfermedad: ST_Slope × Angina', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Tasa de Enfermedad (%)')
axes[1].set_xlabel('ST_Slope')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5️⃣ Detección de Anomalías

### 🔍 Pregunta 15: ¿Existen valores atípicos en las mediciones clínicas?

In [ ]:
print("="*80)
print("DETECCIÓN DE OUTLIERS")
print("="*80)

# Análisis de outliers por variable numérica
numeric_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']

def detect_outliers_iqr(data, column):
    """Detecta outliers usando el método IQR"""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

outlier_summary = []

for col in numeric_cols:
    if col == 'Cholesterol':
        data_temp = df[df[col] > 0]  # Excluir valores 0
    elif col == 'RestingBP':
        data_temp = df[df[col] > 0]  # Excluir valores 0
    else:
        data_temp = df
    
    outliers, lower, upper = detect_outliers_iqr(data_temp, col)
    pct_outliers = (len(outliers) / len(data_temp)) * 100
    
    outlier_summary.append({
        'Variable': col,
        'N Outliers': len(outliers),
        'Porcentaje (%)': pct_outliers,
        'Límite Inferior': lower,
        'Límite Superior': upper
    })

outlier_df = pd.DataFrame(outlier_summary)
print("\n📊 Resumen de Outliers por Variable:\n")
print(outlier_df.to_string(index=False))

In [ ]:
# Visualización de outliers
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    if col == 'Cholesterol':
        data_plot = df[df[col] > 0]
    elif col == 'RestingBP':
        data_plot = df[df[col] > 0]
    else:
        data_plot = df
    
    # Boxplot con outliers destacados
    bp = axes[idx].boxplot(data_plot[col], vert=True, patch_artist=True,
                           boxprops=dict(facecolor='#3498db', alpha=0.7),
                           medianprops=dict(color='red', linewidth=2),
                           flierprops=dict(marker='o', markerfacecolor='red', 
                                          markersize=8, alpha=0.5))
    axes[idx].set_title(f'Outliers en {col}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel(col)
    axes[idx].grid(alpha=0.3)

# Eliminar el subplot extra
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

### 🔍 Pregunta 16: ¿Estos outliers corresponden a casos reales o errores de medición?

In [ ]:
print("="*80)
print("ANÁLISIS DETALLADO DE OUTLIERS")
print("="*80)

# Analizar outliers en Colesterol
df_chol = df[df['Cholesterol'] > 0]
chol_outliers, chol_lower, chol_upper = detect_outliers_iqr(df_chol, 'Cholesterol')

print(f"\n🔍 OUTLIERS EN COLESTEROL:")
print(f"   • Límite superior: {chol_upper:.1f} mg/dl")
print(f"   • Valores extremos encontrados: {len(chol_outliers)}")
if len(chol_outliers) > 0:
    print(f"   • Rango de outliers: {chol_outliers['Cholesterol'].min():.0f} - {chol_outliers['Cholesterol'].max():.0f} mg/dl")
    print(f"\n   📊 Distribución de enfermedad en outliers de colesterol:")
    print(f"      - Con enfermedad: {(chol_outliers['HeartDisease'] == 1).sum()}")
    print(f"      - Sin enfermedad: {(chol_outliers['HeartDisease'] == 0).sum()}")
    print(f"      - Tasa: {chol_outliers['HeartDisease'].mean() * 100:.1f}%")
    print("\n   ✅ Estos valores son médicamente posibles (hipercolesterolemia)")

# Analizar outliers en MaxHR
maxhr_outliers, maxhr_lower, maxhr_upper = detect_outliers_iqr(df, 'MaxHR')

print(f"\n🔍 OUTLIERS EN MaxHR:")
print(f"   • Límite inferior: {maxhr_lower:.1f} bpm")
print(f"   • Límite superior: {maxhr_upper:.1f} bpm")
print(f"   • Valores extremos encontrados: {len(maxhr_outliers)}")
if len(maxhr_outliers) > 0:
    print(f"   • Rango de outliers: {maxhr_outliers['MaxHR'].min():.0f} - {maxhr_outliers['MaxHR'].max():.0f} bpm")
    print(f"\n   📊 Distribución de enfermedad en outliers de MaxHR:")
    print(f"      - Tasa: {maxhr_outliers['HeartDisease'].mean() * 100:.1f}%")

# Analizar outliers en Oldpeak
oldpeak_outliers, oldpeak_lower, oldpeak_upper = detect_outliers_iqr(df, 'Oldpeak')

print(f"\n🔍 OUTLIERS EN Oldpeak:")
print(f"   • Límite superior: {oldpeak_upper:.2f}")
print(f"   • Valores extremos encontrados: {len(oldpeak_outliers)}")
if len(oldpeak_outliers) > 0:
    print(f"   • Rango de outliers: {oldpeak_outliers['Oldpeak'].min():.2f} - {oldpeak_outliers['Oldpeak'].max():.2f}")
    print(f"   • Tasa de enfermedad en outliers: {oldpeak_outliers['HeartDisease'].mean() * 100:.1f}%")

### 🔍 Pregunta 17: ¿Cómo debemos tratar estos valores extremos?

In [ ]:
print("="*80)
print("RECOMENDACIONES PARA TRATAMIENTO DE OUTLIERS")
print("="*80)

print("""
📋 ESTRATEGIA PROPUESTA PARA OUTLIERS:

1️⃣ VALORES DE 0 EN COLESTEROL Y PRESIÓN ARTERIAL:
   ❌ Acción: ELIMINAR o IMPUTAR
   → Son médicamente imposibles, probablemente datos faltantes
   → Opciones:
      a) Eliminar filas (si son pocas)
      b) Imputar con mediana por grupo (HeartDisease)
      c) Usar KNN imputation

2️⃣ OUTLIERS EN COLESTEROL ALTO (>400 mg/dl):
   ✅ Acción: MANTENER
   → Son valores médicamente posibles (hipercolesterolemia severa)
   → Pueden ser informativos para el modelo
   → Considerar transformación logarítmica si afecta al modelo

3️⃣ OUTLIERS EN MaxHR (muy bajos <80 bpm):
   ⚠️ Acción: REVISAR
   → Pueden ser casos reales (bradicardia)
   → Verificar si hay patrones asociados
   → Mantener pero monitorear impacto en modelo

4️⃣ OUTLIERS EN Oldpeak (muy altos >4):
   ✅ Acción: MANTENER
   → Son mediciones válidas de depresión del ST
   → Altamente predictivos de enfermedad
   → Considerar winsorización si es necesario

5️⃣ OUTLIERS EN RestingBP:
   ⚠️ Acción: REVISAR CASO POR CASO
   → Valores muy bajos (<90) o muy altos (>200) revisar
   → Pueden ser errores de medición o casos especiales

🎯 TÉCNICAS RECOMENDADAS:
   • IQR method para detección inicial
   • Z-score para outliers multivariados
   • Isolation Forest para detección avanzada
   • Visualización con boxplots y scatter plots
""")

# Identificar filas con múltiples problemas
problematic_rows = df[
    (df['Cholesterol'] == 0) | 
    (df['RestingBP'] == 0)
]

print(f"\n⚠️ FILAS PROBLEMÁTICAS IDENTIFICADAS:")
print(f"   • Total: {len(problematic_rows)} ({len(problematic_rows)/len(df)*100:.2f}%)")
print(f"   • Colesterol = 0: {(df['Cholesterol'] == 0).sum()}")
print(f"   • RestingBP = 0: {(df['RestingBP'] == 0).sum()}")

if len(problematic_rows) > 0:
    print("\n📋 Primeras filas problemáticas:")
    print(problematic_rows.head())

In [ ]:
# Visualización de outliers multivariados
from sklearn.ensemble import IsolationForest

# Preparar datos para Isolation Forest
df_if = df[df['Cholesterol'] > 0][['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']].copy()

# Entrenar Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42)
outlier_predictions = iso_forest.fit_predict(df_if)

# Añadir predicciones
df_if['Outlier'] = outlier_predictions
df_if['HeartDisease'] = df[df['Cholesterol'] > 0]['HeartDisease'].values

n_outliers = (df_if['Outlier'] == -1).sum()
print(f"\n🤖 ISOLATION FOREST - Outliers Multivariados:")
print(f"   • Outliers detectados: {n_outliers} ({n_outliers/len(df_if)*100:.2f}%)")
print(f"   • Tasa de enfermedad en outliers: {df_if[df_if['Outlier'] == -1]['HeartDisease'].mean() * 100:.1f}%")
print(f"   • Tasa de enfermedad en normales: {df_if[df_if['Outlier'] == 1]['HeartDisease'].mean() * 100:.1f}%")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter Age vs Cholesterol
normal_points = df_if[df_if['Outlier'] == 1]
outlier_points = df_if[df_if['Outlier'] == -1]

axes[0].scatter(normal_points['Age'], normal_points['Cholesterol'],
                c='#3498db', alpha=0.5, s=50, label='Normal', edgecolors='black', linewidth=0.5)
axes[0].scatter(outlier_points['Age'], outlier_points['Cholesterol'],
                c='#e74c3c', alpha=0.8, s=100, label='Outlier', marker='^', 
                edgecolors='black', linewidth=1)
axes[0].set_title('Detección de Outliers: Edad vs Colesterol', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Edad (años)')
axes[0].set_ylabel('Colesterol (mg/dl)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Scatter MaxHR vs Oldpeak
axes[1].scatter(normal_points['MaxHR'], normal_points['Oldpeak'],
                c='#3498db', alpha=0.5, s=50, label='Normal', edgecolors='black', linewidth=0.5)
axes[1].scatter(outlier_points['MaxHR'], outlier_points['Oldpeak'],
                c='#e74c3c', alpha=0.8, s=100, label='Outlier', marker='^',
                edgecolors='black', linewidth=1)
axes[1].set_title('Detección de Outliers: MaxHR vs Oldpeak', fontsize=13, fontweight='bold')
axes[1].set_xlabel('MaxHR (bpm)')
axes[1].set_ylabel('Oldpeak')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6️⃣ Análisis Adicional: Variables Categóricas

In [ ]:
# Análisis de ECG en reposo
print("="*80)
print("ANÁLISIS DE VARIABLES CATEGÓRICAS ADICIONALES")
print("="*80)

print("\n📊 RESTING ECG:")
ecg_disease = pd.crosstab(df['RestingECG'], df['HeartDisease'], normalize='index') * 100
ecg_disease.columns = ['Sin Enfermedad (%)', 'Con Enfermedad (%)']
print(ecg_disease.round(2))

print("\n📊 ST_SLOPE:")
st_disease = pd.crosstab(df['ST_Slope'], df['HeartDisease'], normalize='index') * 100
st_disease.columns = ['Sin Enfermedad (%)', 'Con Enfermedad (%)']
print(st_disease.round(2))

print("\n📊 EXERCISE ANGINA:")
angina_disease = pd.crosstab(df['ExerciseAngina'], df['HeartDisease'], normalize='index') * 100
angina_disease.columns = ['Sin Enfermedad (%)', 'Con Enfermedad (%)']
angina_disease.index = ['Sin Angina', 'Con Angina']
print(angina_disease.round(2))

In [ ]:
# Visualización de variables categóricas
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. RestingECG
ecg_counts = pd.crosstab(df['RestingECG'], df['HeartDisease'])
ecg_counts.plot(kind='bar', ax=axes[0, 0], color=['#2ecc71', '#e74c3c'],
                alpha=0.8, edgecolor='black')
axes[0, 0].set_title('RestingECG por Diagnóstico', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('RestingECG')
axes[0, 0].set_ylabel('Número de Pacientes')
axes[0, 0].legend(['Sin Enfermedad', 'Con Enfermedad'])
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=45)
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. ST_Slope
st_counts = pd.crosstab(df['ST_Slope'], df['HeartDisease'])
st_counts.plot(kind='bar', ax=axes[0, 1], color=['#2ecc71', '#e74c3c'],
               alpha=0.8, edgecolor='black')
axes[0, 1].set_title('ST_Slope por Diagnóstico', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('ST_Slope')
axes[0, 1].set_ylabel('Número de Pacientes')
axes[0, 1].legend(['Sin Enfermedad', 'Con Enfermedad'])
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=45)
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. ExerciseAngina
angina_counts = pd.crosstab(df['ExerciseAngina'], df['HeartDisease'])
angina_counts.index = ['Sin Angina', 'Con Angina']
angina_counts.plot(kind='bar', ax=axes[1, 0], color=['#2ecc71', '#e74c3c'],
                   alpha=0.8, edgecolor='black')
axes[1, 0].set_title('ExerciseAngina por Diagnóstico', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('ExerciseAngina')
axes[1, 0].set_ylabel('Número de Pacientes')
axes[1, 0].legend(['Sin Enfermedad', 'Con Enfermedad'])
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45)
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. Resumen de tasas
categorical_vars = ['ChestPainType', 'RestingECG', 'ST_Slope', 'ExerciseAngina']
disease_rates = []

for var in categorical_vars:
    rate = df.groupby(var)['HeartDisease'].mean().max() * 100
    disease_rates.append(rate)

axes[1, 1].barh(categorical_vars, disease_rates, color='#e67e22', alpha=0.8, edgecolor='black')
axes[1, 1].set_title('Máxima Tasa de Enfermedad por Variable Categórica', 
                     fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Tasa Máxima de Enfermedad (%)')
axes[1, 1].grid(axis='x', alpha=0.3)

for i, val in enumerate(disease_rates):
    axes[1, 1].text(val + 1, i, f'{val:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.show()

---
## 7️⃣ Conclusiones y Recomendaciones

In [ ]:
print("="*80)
print("RESUMEN EJECUTIVO DEL EDA")
print("="*80)

print("""
🎯 HALLAZGOS PRINCIPALES:

1️⃣ BALANCE DE CLASES:
   • Dataset relativamente balanceado
   • No se requieren técnicas agresivas de balanceo

2️⃣ DATOS FALTANTES:
   • Valores de 0 en Colesterol y RestingBP probablemente son datos faltantes
   • Se requiere tratamiento antes del modelado

3️⃣ VARIABLES MÁS PREDICTIVAS (basado en correlación):
   • ST_Slope (especialmente "Flat")
   • ExerciseAngina (Sí/No)
   • Oldpeak (depresión del ST)
   • ChestPainType (especialmente "ASY" - Asintomático)
   • MaxHR (frecuencia cardíaca máxima)

4️⃣ DIFERENCIAS POR SEXO:
   • Los hombres tienen mayor tasa de enfermedad
   • Patrones de síntomas diferentes entre sexos
   • Considerar sexo como feature importante

5️⃣ EDAD:
   • Factor de riesgo significativo
   • Pacientes con enfermedad tienden a ser mayores
   • Interacción con otras variables (colesterol, presión)

6️⃣ DIABETES (FastingBS):
   • Asociación moderada con enfermedad cardíaca
   • Diabéticos tienen perfiles de riesgo diferentes

7️⃣ OUTLIERS:
   • Mayoría son valores médicamente posibles
   • No eliminar sin análisis detallado
   • Considerar transformaciones si afectan al modelo

""")

print("="*80)
print("RECOMENDACIONES PARA PREPROCESAMIENTO")
print("="*80)

print("""
🔧 PASOS SUGERIDOS PARA PREPARACIÓN DE DATOS:

1. LIMPIEZA:
   ✓ Tratar valores de 0 en Colesterol y RestingBP (imputación o eliminación)
   ✓ Verificar y eliminar duplicados si existen
   ✓ Validar rangos de valores numéricos

2. TRANSFORMACIONES:
   ✓ One-Hot Encoding para variables categóricas
   ✓ Considerar escalado (StandardScaler o MinMaxScaler)
   ✓ Evaluar transformación logarítmica para variables asimétricas

3. FEATURE ENGINEERING:
   ✓ Crear features de interacción (Edad × Colesterol)
   ✓ Categorizar variables continuas en rangos clínicos
   ✓ Crear score de riesgo combinado

4. SELECCIÓN DE FEATURES:
   ✓ Priorizar variables con alta correlación
   ✓ Evaluar multicolinealidad
   ✓ Usar técnicas de selección (RFE, Feature Importance)

5. VALIDACIÓN:
   ✓ Stratified K-Fold para mantener proporción de clases
   ✓ Train-test split con estratificación
   ✓ Validación cruzada para estabilidad

""")

# Guardar dataset limpio sugerido
print("\n💾 Preparando dataset para exportación...")
df_clean = df[(df['Cholesterol'] > 0) & (df['RestingBP'] > 0)].copy()
print(f"   • Filas originales: {len(df)}")
print(f"   • Filas después de limpieza: {len(df_clean)}")
print(f"   • Filas eliminadas: {len(df) - len(df_clean)}")

# Guardar
output_path = '../data/heart_cleaned_p.csv'
df_clean.to_csv(output_path, index=False)
print(f"\n✅ Dataset limpio guardado en: {output_path}")

---
## 📈 Resumen Estadístico Final

In [ ]:
# Estadísticas descriptivas completas
print("="*80)
print("ESTADÍSTICAS DESCRIPTIVAS COMPLETAS")
print("="*80)

print("\n📊 Variables Numéricas:\n")
print(df[numeric_cols].describe().round(2))

print("\n📊 Variables Categóricas:\n")
categorical_cols = ['Sex', 'ChestPainType', 'FastingBS', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())
    print(f"Valores únicos: {df[col].nunique()}")

---
## 🎓 Conclusiones Finales del EDA

### ✅ Respuestas a Preguntas de Investigación:

#### **3.1 Exploración General:**
- ✓ Dataset balanceado (~55% con enfermedad, ~45% sin enfermedad)
- ✓ Población predominantemente masculina
- ✓ Rango de edad: 28-77 años, media ~54 años
- ✓ Datos faltantes codificados como 0 en Colesterol y RestingBP

#### **3.2 Variables Individuales:**
- ✓ Edad: Diferencia significativa (pacientes enfermos son mayores)
- ✓ Dolor torácico ASY (Asintomático) muy común en enfermos
- ✓ Colesterol: Diferencias moderadas entre grupos
- ✓ RestingBP: Diferencias leves
- ✓ FastingBS: Asociación moderada con enfermedad
- ✓ MaxHR: Diferencia significativa (menor en enfermos)

#### **3.3 Relaciones entre Variables:**
- ✓ ST_Slope es la variable más predictiva
- ✓ ExerciseAngina fuertemente asociada con enfermedad
- ✓ Patrones diferentes entre hombres y mujeres
- ✓ Edad + Colesterol alto = mayor riesgo
- ✓ Diabéticos tienen perfil de riesgo diferente
- ✓ Múltiples interacciones significativas identificadas

#### **3.4 Anomalías:**
- ✓ Outliers en Colesterol son valores reales (hipercolesterolemia)
- ✓ Outliers en Oldpeak son informativos
- ✓ Valores de 0 deben tratarse como faltantes
- ✓ Aplicar Isolation Forest para detección multivariada

### 🚀 Próximos Pasos:

1. **Preprocesamiento**: Tratar valores 0, imputación, encoding
2. **Feature Engineering**: Crear variables de interacción
3. **Modelado**: Probar múltiples algoritmos (Logistic Regression, Random Forest, XGBoost, etc.)
4. **Evaluación**: Usar métricas apropiadas (AUC-ROC, F1-Score, Precision, Recall)
5. **Optimización**: Hyperparameter tuning y validación cruzada

---

**📝 Autor:** Miembro P  
**📅 Fecha:** Análisis EDA Completo  
**✅ Estado:** Listo para fase de entrenamiento

---
## 🏥 Análisis de Supervivencia (Survival Analysis)

El análisis de supervivencia es una técnica estadística avanzada para estudiar el tiempo hasta que ocurre un evento (en este caso, la enfermedad cardíaca). Aunque nuestro dataset es de corte transversal, podemos usar la **edad** como proxy del tiempo y aplicar técnicas de supervivencia para entender mejor los patrones de riesgo.

### Objetivos:
1. Estimar curvas de supervivencia (Kaplan-Meier)
2. Comparar supervivencia entre grupos (sexo, diabetes, dolor torácico)
3. Identificar factores de riesgo con Cox Proportional Hazards Model
4. Visualizar el riesgo acumulado

In [ ]:
# Importar librerías de análisis de supervivencia
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines.plotting import plot_lifetimes

print("="*80)
print("🏥 ANÁLISIS DE SUPERVIVENCIA")
print("="*80)

# Preparar datos para análisis de supervivencia
# Usamos la edad como tiempo y HeartDisease como evento
df_survival = df.copy()
df_survival['time'] = df_survival['Age']  # Edad como tiempo
df_survival['event'] = df_survival['HeartDisease']  # Enfermedad como evento

# Eliminar valores anómalos para mejor análisis
df_survival_clean = df_survival[(df_survival['Cholesterol'] > 0) & 
                                 (df_survival['RestingBP'] > 0)].copy()

print(f"\n📊 Datos preparados para análisis de supervivencia:")
print(f"   • Total de observaciones: {len(df_survival_clean)}")
print(f"   • Eventos (enfermedades): {df_survival_clean['event'].sum()}")
print(f"   • Sin eventos: {len(df_survival_clean) - df_survival_clean['event'].sum()}")
print(f"   • Tasa de eventos: {df_survival_clean['event'].mean()*100:.2f}%")

### 📈 Curva de Kaplan-Meier Global

La curva de Kaplan-Meier estima la probabilidad de "supervivencia" (no tener enfermedad cardíaca) a diferentes edades.

In [ ]:
# Ajustar modelo Kaplan-Meier global
kmf = KaplanMeierFitter()
kmf.fit(durations=df_survival_clean['time'], 
        event_observed=df_survival_clean['event'],
        label='Población General')

# Visualización
fig, ax = plt.subplots(figsize=(12, 6))
kmf.plot_survival_function(ax=ax, ci_show=True)
ax.set_title('Curva de Kaplan-Meier - Probabilidad de No Tener Enfermedad Cardíaca', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Edad (años)', fontsize=12)
ax.set_ylabel('Probabilidad de Supervivencia (Sin Enfermedad)', fontsize=12)
ax.grid(alpha=0.3)
ax.legend(loc='best', fontsize=11)

# Añadir línea en mediana de supervivencia
median_survival = kmf.median_survival_time_
if not np.isnan(median_survival):
    ax.axvline(median_survival, color='red', linestyle='--', linewidth=2, 
               label=f'Edad Mediana: {median_survival:.1f} años')
    ax.legend(loc='best', fontsize=11)

plt.tight_layout()
plt.show()

# Estadísticas de supervivencia
print("\n📊 Estadísticas de Supervivencia:")
print(f"   • Edad mediana de aparición de enfermedad: {median_survival:.1f} años" if not np.isnan(median_survival) else "   • Edad mediana: No alcanzada")
print(f"   • Probabilidad de no tener enfermedad a los 40 años: {kmf.survival_function_at_times(40).values[0]:.2%}")
print(f"   • Probabilidad de no tener enfermedad a los 50 años: {kmf.survival_function_at_times(50).values[0]:.2%}")
print(f"   • Probabilidad de no tener enfermedad a los 60 años: {kmf.survival_function_at_times(60).values[0]:.2%}")
print(f"   • Probabilidad de no tener enfermedad a los 70 años: {kmf.survival_function_at_times(70).values[0]:.2%}")

### 👥 Comparación de Supervivencia por Sexo

Comparamos las curvas de supervivencia entre hombres y mujeres para identificar diferencias en el riesgo de enfermedad cardíaca.

In [ ]:
# Ajustar curvas por sexo
fig, ax = plt.subplots(figsize=(12, 6))

for sex in df_survival_clean['Sex'].unique():
    kmf_sex = KaplanMeierFitter()
    mask = df_survival_clean['Sex'] == sex
    label = 'Masculino' if sex == 'M' else 'Femenino'
    
    kmf_sex.fit(durations=df_survival_clean[mask]['time'],
                event_observed=df_survival_clean[mask]['event'],
                label=label)
    
    kmf_sex.plot_survival_function(ax=ax, ci_show=True)

ax.set_title('Curvas de Kaplan-Meier por Sexo', fontsize=14, fontweight='bold')
ax.set_xlabel('Edad (años)', fontsize=12)
ax.set_ylabel('Probabilidad de Supervivencia (Sin Enfermedad)', fontsize=12)
ax.grid(alpha=0.3)
ax.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

# Test log-rank para comparar grupos
male_data = df_survival_clean[df_survival_clean['Sex'] == 'M']
female_data = df_survival_clean[df_survival_clean['Sex'] == 'F']

results = logrank_test(male_data['time'], female_data['time'],
                       male_data['event'], female_data['event'])

print("\n📊 Test Log-Rank (Comparación entre Sexos):")
print(f"   • Estadístico: {results.test_statistic:.4f}")
print(f"   • p-valor: {results.p_value:.4f}")

if results.p_value < 0.05:
    print("   ✅ HAY diferencia significativa en supervivencia entre sexos (p < 0.05)")
else:
    print("   ❌ NO hay diferencia significativa en supervivencia entre sexos (p >= 0.05)")

### 💔 Comparación de Supervivencia por Tipo de Dolor Torácico

El tipo de dolor torácico es un factor importante en el diagnóstico de enfermedad cardíaca.

In [ ]:
# Ajustar curvas por tipo de dolor torácico
fig, ax = plt.subplots(figsize=(14, 7))

chest_pain_labels = {
    'ATA': 'Angina Atípica',
    'NAP': 'Dolor No Anginoso',
    'ASY': 'Asintomático',
    'TA': 'Angina Típica'
}

colors_chest = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']

for i, chest_type in enumerate(df_survival_clean['ChestPainType'].unique()):
    kmf_chest = KaplanMeierFitter()
    mask = df_survival_clean['ChestPainType'] == chest_type
    
    kmf_chest.fit(durations=df_survival_clean[mask]['time'],
                  event_observed=df_survival_clean[mask]['event'],
                  label=chest_pain_labels.get(chest_type, chest_type))
    
    kmf_chest.plot_survival_function(ax=ax, ci_show=False, color=colors_chest[i % len(colors_chest)])

ax.set_title('Curvas de Kaplan-Meier por Tipo de Dolor Torácico', fontsize=14, fontweight='bold')
ax.set_xlabel('Edad (años)', fontsize=12)
ax.set_ylabel('Probabilidad de Supervivencia (Sin Enfermedad)', fontsize=12)
ax.grid(alpha=0.3)
ax.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

print("\n💡 Interpretación:")
print("   • Pacientes ASINTOMÁTICOS (ASY) tienen mayor riesgo de enfermedad cardíaca")
print("   • El tipo de dolor torácico es un predictor importante del riesgo")

### 📊 Modelo de Riesgos Proporcionales de Cox

El modelo de Cox identifica qué factores aumentan o disminuyen el riesgo de enfermedad cardíaca, controlando por múltiples variables simultáneamente.

In [ ]:
# Preparar datos para modelo Cox
df_cox = df_survival_clean.copy()

# Codificar variables categóricas
df_cox['Sex_M'] = (df_cox['Sex'] == 'M').astype(int)
df_cox['ChestPain_ASY'] = (df_cox['ChestPainType'] == 'ASY').astype(int)
df_cox['ExerciseAngina_Y'] = (df_cox['ExerciseAngina'] == 'Y').astype(int)
df_cox['ST_Slope_Flat'] = (df_cox['ST_Slope'] == 'Flat').astype(int)
df_cox['ST_Slope_Down'] = (df_cox['ST_Slope'] == 'Down').astype(int)

# Seleccionar variables para el modelo
cox_vars = ['Sex_M', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR',
            'ChestPain_ASY', 'ExerciseAngina_Y', 'Oldpeak', 
            'ST_Slope_Flat', 'ST_Slope_Down', 'time', 'event']

df_cox_model = df_cox[cox_vars].copy()

# Ajustar modelo Cox
cph = CoxPHFitter()
cph.fit(df_cox_model, duration_col='time', event_col='event')

# Mostrar resultados
print("="*80)
print("MODELO DE RIESGOS PROPORCIONALES DE COX")
print("="*80)
print("\n📊 Resumen del Modelo:")
print(cph.summary[['coef', 'exp(coef)', 'p']])

# Visualizar hazard ratios
fig, ax = plt.subplots(figsize=(10, 8))
cph.plot(ax=ax)
ax.set_title('Hazard Ratios (Razones de Riesgo)', fontsize=14, fontweight='bold')
ax.axvline(1, color='red', linestyle='--', linewidth=2, label='Sin efecto (HR=1)')
ax.set_xlabel('Hazard Ratio (Razón de Riesgo)', fontsize=12)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Interpretación de Hazard Ratios (HR):")
print("   • HR > 1: Factor de RIESGO (aumenta probabilidad de enfermedad)")
print("   • HR < 1: Factor PROTECTOR (disminuye probabilidad de enfermedad)")
print("   • HR = 1: Sin efecto")

# Identificar factores más importantes
significant_factors = cph.summary[cph.summary['p'] < 0.05].sort_values('p')
print(f"\n✅ Factores Significativos (p < 0.05): {len(significant_factors)}")
if len(significant_factors) > 0:
    print("\nTop 5 factores más significativos:")
    for idx, (var, row) in enumerate(significant_factors.head().iterrows(), 1):
        hr = row['exp(coef)']
        p_val = row['p']
        effect = "RIESGO ⬆️" if hr > 1 else "PROTECTOR ⬇️"
        print(f"   {idx}. {var}: HR={hr:.3f}, p={p_val:.4f} → {effect}")

### 📝 Conclusiones del Análisis de Supervivencia

**Hallazgos Clave:**

1. **Curva Global**: La probabilidad de no tener enfermedad cardíaca disminuye con la edad, especialmente después de los 50 años.

2. **Diferencias por Sexo**: Se observan diferencias en las curvas de supervivencia entre hombres y mujeres (validado con test log-rank).

3. **Tipo de Dolor Torácico**: Los pacientes asintomáticos (ASY) muestran mayor riesgo de enfermedad cardíaca, lo que es un hallazgo clínicamente importante.

4. **Factores de Riesgo Principales** (según modelo Cox):
   - Ejercicio con angina (ExerciseAngina_Y)
   - Tipo de pendiente ST (ST_Slope)
   - Dolor torácico asintomático (ChestPain_ASY)
   - Depresión del ST (Oldpeak)

5. **Implicaciones Clínicas**: El análisis de supervivencia confirma que múltiples factores contribuyen al riesgo cardiovascular, y algunos factores "silenciosos" (como ser asintomático) pueden ser igual o más importantes que síntomas evidentes.

---